# Global Topic Renaming — SynBio Papers (Part 2)

The per-cluster naming in Part 1 works locally: each topic is named in
isolation. This can produce **duplicate or ambiguous names** when two clusters
cover related sub-themes (e.g. both named "History of Synthetic Biology").

This notebook fixes that by giving the LLM a **global view** of all
**SynBio Papers** topics at once. We use **OpenAI function calling** so the
model returns a structured array of `(topic_id, name)` pairs — one per cluster —
guaranteeing distinct, publication-ready names.

Overwrites `papers_topic_names.txt`, adding a `global_name` column.

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 03-topic_names/, where the aux/ package
# resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
from aux.paths import MODELS_DIR, OPENAI_MODEL
from aux.openai_client import load_prompts, make_client
from aux.tables import load_topic_names, save_topic_names
from aux.global_rename import rename_topics_global

# ── CONFIG: SynBio Papers ──────────────────────────────────────────────────
PREFIX = "papers"
MODEL  = OPENAI_MODEL

prompts = load_prompts()
client = make_client()

## 1. Load Part 1 results

In [3]:
names = load_topic_names(PREFIX)
print(f"Papers: {len(names)} topics")
names[["topic", "name", "description"]].head()

Papers: 225 topics


,topic,name,description
0,0,Synthetic Biology in Plant Engineering,This cluster centers on the advancement of too...
1,1,Ethics and Society in Synthetic Biology,"This cluster centers on the ethical, social, a..."
2,2,Plant Biosynthesis Engineering,This cluster focuses on leveraging synthetic b...
3,3,"Synthetic Biology for Diagnostics, Monitoring,...",This cluster centers on harnessing synthetic b...
4,4,CRISPR-Cas Synthetic Biology,This cluster centers on the utilization of CRI...


## 2. Global rename (function calling)

In [4]:
renamed = rename_topics_global(names, client, prompts, model=MODEL)
renamed[["topic", "name", "global_name", "description"]]

,topic,name,global_name,description
0,0,Synthetic Biology in Plant Engineering,Plant Genetic Engineering Tools,This cluster centers on the advancement of too...
1,1,Ethics and Society in Synthetic Biology,Ethics and Society in Synthetic Biology,"This cluster centers on the ethical, social, a..."
2,2,Plant Biosynthesis Engineering,Plant Biosynthesis Pathway Engineering,This cluster focuses on leveraging synthetic b...
3,3,"Synthetic Biology for Diagnostics, Monitoring,...",Synthetic Biology for Diagnostics and Therapeu...,This cluster centers on harnessing synthetic b...
4,4,CRISPR-Cas Synthetic Biology,CRISPR-Cas Systems in Synthetic Biology,This cluster centers on the utilization of CRI...
...,...,...,...,...
220,220,Microfluidics in Synthetic Biology,Microfluidics in Synthetic Biology,This cluster focuses on leveraging microfluidi...
221,221,Synthetic Nucleic Acid Vaccines,Synthetic Nucleic Acid Vaccines,This cluster centers on the utilization of syn...
222,222,Viral Vector Engineering,Viral Vector Engineering,This collection of texts focuses on utilizing ...
223,223,Bioelectrical and Mechanical Engineering in Sy...,Bioelectrical and Mechanical Systems,The core focus of this cluster is the integrat...


## 3. Save final results

In [5]:
save_topic_names(renamed, PREFIX)
print(f"Saved → {MODELS_DIR / f'{PREFIX}_topic_names.txt'} (added global_name)")

Saved → /Users/cristian/Desktop/GitHub/igem-synbio/assets/topic_models/papers_topic_names.txt (added global_name)
